In [1]:
# calc_chrom_arm_length.ipynb - calculate lengths of chromosome arms.

In [2]:
import numpy as np
import pandas as pd

In [3]:
in_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/dev/HCC3N_600spot/data/refseq/cytoBand.txt"
out_chrom_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/dev/HCC3N_600spot/data/refseq/chrom_length.tsv"
out_arm_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/dev/HCC3N_600spot/data/refseq/chrom_arm_length.tsv"

## Load data

In [4]:
df = pd.read_csv(in_fn, sep = '\t', header = None)
df.columns = ['chrom', 'start', 'end', 'band', 'staining']
df

,chrom,start,end,band,staining
0,chr1,0,2300000,p36.33,gneg
1,chr1,2300000,5300000,p36.32,gpos25
2,chr1,5300000,7100000,p36.31,gneg
3,chr1,7100000,9100000,p36.23,gpos25
4,chr1,9100000,12500000,p36.22,gneg
...,...,...,...,...,...
1544,chr19_MU273387v1_alt,0,89211,NaN,gneg
1545,chr16_MU273376v1_fix,0,87715,NaN,gneg
1546,chrX_MU273393v1_fix,0,68810,NaN,gneg
1547,chr8_MU273360v1_fix,0,39290,NaN,gneg


In [5]:
def format_chrom(x):
    x = str(x)
    return x[3:] if x.startswith('chr') else x

target_chroms = [str(i) for i in range(1, 23)] + ['X', 'Y']
df['chrom'] = df['chrom'].map(format_chrom)
df = df.loc[df['chrom'].isin(target_chroms)].copy()
df

,chrom,start,end,band,staining
0,1,0,2300000,p36.33,gneg
1,1,2300000,5300000,p36.32,gpos25
2,1,5300000,7100000,p36.31,gneg
3,1,7100000,9100000,p36.23,gpos25
4,1,9100000,12500000,p36.22,gneg
...,...,...,...,...,...
1287,Y,12400000,17100000,q11.221,gpos50
1288,Y,17100000,19600000,q11.222,gneg
1289,Y,19600000,23800000,q11.223,gpos50
1290,Y,23800000,26600000,q11.23,gneg


In [6]:
df['arm'] = df['band'].map(lambda x: x[0])
assert np.all(df['arm'].isin(['p', 'q']))
df

,chrom,start,end,band,staining,arm
0,1,0,2300000,p36.33,gneg,p
1,1,2300000,5300000,p36.32,gpos25,p
2,1,5300000,7100000,p36.31,gneg,p
3,1,7100000,9100000,p36.23,gpos25,p
4,1,9100000,12500000,p36.22,gneg,p
...,...,...,...,...,...,...
1287,Y,12400000,17100000,q11.221,gpos50,q
1288,Y,17100000,19600000,q11.222,gneg,q
1289,Y,19600000,23800000,q11.223,gpos50,q
1290,Y,23800000,26600000,q11.23,gneg,q


## Calculate chromosome arm lengths

In [7]:
anno = df.groupby(['chrom', 'arm']).agg({'start':'min', 'end':'max'}).reset_index()
anno.head()

,chrom,arm,start,end
0,1,p,0,123400000
1,1,q,123400000,248956422
2,10,p,0,39800000
3,10,q,39800000,133797422
4,11,p,0,53400000


In [8]:
anno['length'] = anno['end'] - anno['start']
anno.head()

,chrom,arm,start,end,length
0,1,p,0,123400000,123400000
1,1,q,123400000,248956422,125556422
2,10,p,0,39800000,39800000
3,10,q,39800000,133797422,93997422
4,11,p,0,53400000,53400000


In [9]:
order = target_chroms
anno['chrom'] = pd.Categorical(anno['chrom'], categories = order, ordered = True)
anno = anno.sort_values(['chrom', 'arm'])
anno

,chrom,arm,start,end,length
0,1,p,0,123400000,123400000
1,1,q,123400000,248956422,125556422
22,2,p,0,93900000,93900000
23,2,q,93900000,242193529,148293529
30,3,p,0,90900000,90900000
31,3,q,90900000,198295559,107395559
32,4,p,0,50000000,50000000
33,4,q,50000000,190214555,140214555
34,5,p,0,48800000,48800000
35,5,q,48800000,181538259,132738259


In [10]:
anno.to_csv(out_arm_fn, sep = '\t', header = True, index = False)

## Calculate chromosome lengths

In [11]:
anno_chrom = anno.groupby(['chrom']).agg({'start':'min', 'end':'max'}).reset_index()
anno_chrom.head()

/tmp/pbs.1777102.xomics/ipykernel_69720/3603168845.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  anno_chrom = anno.groupby(['chrom']).agg({'start':'min', 'end':'max'}).reset_index()


,chrom,start,end
0,1,0,248956422
1,2,0,242193529
2,3,0,198295559
3,4,0,190214555
4,5,0,181538259


In [12]:
anno_chrom['length'] = anno_chrom['end'] - anno_chrom['start']
anno_chrom

,chrom,start,end,length
0,1,0,248956422,248956422
1,2,0,242193529,242193529
2,3,0,198295559,198295559
3,4,0,190214555,190214555
4,5,0,181538259,181538259
5,6,0,170805979,170805979
6,7,0,159345973,159345973
7,8,0,145138636,145138636
8,9,0,138394717,138394717
9,10,0,133797422,133797422


In [13]:
anno_chrom.to_csv(out_chrom_fn, sep = '\t', header = True, index = False)